# Sarcasm Project Xiangyu Li

I implemented two traditional baseline models to establish the initial performance benchmarks for our sarcasm detection task:

Feature Extraction: Used `TfidfVectorizer` (capped at 10,000 features) to convert the text data into numerical format.

Model 1: A **Majority Vote** classifier (`DummyClassifier`) as the simplest baseline.

Model 2: A **Support Vector Machine** (`LinearSVC`) as the advanced baseline.

In [ ]:
# Install if you do not have the following libiraries
pip install --upgrade pip pandas scikit-learn jupyter matplotlib

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

### 1. Load the dataset

In [37]:
train_df = pd.read_csv('sarcasm_train.csv')
test_df = pd.read_csv('sarcasm_test.csv')

# Extract features and labels. 
# Using .fillna('') to prevent errors caused by empty text rows.
X_train_text = train_df['combined_text'].fillna('')
y_train = train_df['label']

X_test_text = test_df['combined_text'].fillna('')
y_test = test_df['label']

print(f"Data loaded successfully. Training set size: {len(train_df)}, Test set size: {len(test_df)}")

Data loaded successfully. Training set size: 807972, Test set size: 201992


### 2. Convert text to numerical vectors using TF-IDF

In [38]:
# Keep the top 10000 most frequent words to prevent memory overload
vectorizer = TfidfVectorizer(max_features=10000) 

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print("Vectorization complete. Feature matrix shape:", X_train.shape)

Vectorization complete. Feature matrix shape: (807972, 10000)


### 3. Majority Vote Model (Simple Baseline)

In [39]:
# The 'most_frequent' strategy acts as a majority vote classifier
majority_clf = DummyClassifier(strategy='most_frequent')
majority_clf.fit(X_train, y_train)

y_pred_majority = majority_clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_majority))
print("F1 Score (Macro):", f1_score(y_test, y_pred_majority, average='macro'))
print("\nDetailed Classification Report:\n", classification_report(y_test, y_pred_majority, zero_division=0))

Accuracy: 0.4979405124955444
F1 Score (Macro): 0.3324167470882963

Detailed Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00    101412
           1       0.50      1.00      0.66    100580

    accuracy                           0.50    201992
   macro avg       0.25      0.50      0.33    201992
weighted avg       0.25      0.50      0.33    201992



### 4. Support Vector Machine Model

In [40]:
# LinearSVC is highly optimized and much faster for large-scale text data
svm_clf = LinearSVC(random_state=42, dual=False) 
svm_clf.fit(X_train, y_train)

y_pred_svm = svm_clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("F1 Score (Macro):", f1_score(y_test, y_pred_svm, average='macro'))
print("\nDetailed Classification Report:\n", classification_report(y_test, y_pred_svm))

Accuracy: 0.6601301041625411
F1 Score (Macro): 0.6595727657765171

Detailed Classification Report:
               precision    recall  f1-score   support

           0       0.65      0.70      0.67    101412
           1       0.67      0.62      0.65    100580

    accuracy                           0.66    201992
   macro avg       0.66      0.66      0.66    201992
weighted avg       0.66      0.66      0.66    201992



### 5. Diagnostic Analysis: Feature Importance

In [41]:
# Get all 10000 words extracted by the TF-IDF vectorizer
feature_names = vectorizer.get_feature_names_out()

# Get the weight of each word assigned by the SVM model
svm_weights = svm_clf.coef_[0]

# Zip the words with their corresponding weights and sort them
word_weights = list(zip(feature_names, svm_weights))

# Higher positive weights strongly indicate Sarcasm
sarcasm_top20 = sorted(word_weights, key=lambda x: x[1], reverse=True)[:20]
# Lower negative weights strongly indicate Normal (Non-Sarcastic) comments
normal_top20 = sorted(word_weights, key=lambda x: x[1])[:20]

print("Top 20 words most indicative of [Sarcasm]:")
for word, weight in sarcasm_top20:
    print(f"{word}: {weight:.4f}")

print("\nTop 20 words most indicative of [Normal Comments]:")
for word, weight in normal_top20:
    print(f"{word}: {weight:.4f}")

Top 20 words most indicative of [Sarcasm]:
obviously: 2.6920
forgot: 2.6460
clearly: 2.3612
duh: 2.2845
yeah: 2.2435
totally: 2.1886
dropped: 2.1019
bootstraps: 2.0640
gee: 2.0549
dare: 1.9535
shitlord: 1.8595
sheeple: 1.7868
pfft: 1.6742
unbiased: 1.6713
because: 1.6360
scrub: 1.5967
insightful: 1.5882
fault: 1.5802
shocker: 1.5260
sarcasm: 1.5147

Top 20 words most indicative of [Normal Comments]:
iirc: -1.3547
lf: -1.1535
afaik: -1.1296
roast: -1.0952
bingo: -1.0471
psn: -0.9074
esque: -0.8647
mutually: -0.8517
necessarily: -0.8462
generally: -0.8327
theoretically: -0.8265
mainly: -0.8163
imo: -0.8013
kek: -0.8006
premise: -0.7968
gather: -0.7848
depending: -0.7814
tension: -0.7730
essentially: -0.7615
spicy: -0.7568


### 6. Error Analysis: Examining Failure Cases

In [42]:
# Find the indices of all incorrect predictions
incorrect_indices = np.where(y_pred_svm != y_test)[0]

print(f"The SVM model misclassified a total of {len(incorrect_indices)} comments.\n")
print("Here are 5 typical failure cases:\n")

# Display the first 5 incorrect cases for manual inspection
for i in incorrect_indices[:5]:
    actual_label = y_test.iloc[i]
    predicted_label = y_pred_svm[i]
    text = X_test_text.iloc[i]
    
    # Map binary labels back to readable strings (1 = Sarcastic, 0 = Normal)
    actual_str = "Sarcastic" if actual_label == 1 else "Normal"
    pred_str = "Sarcastic" if predicted_label == 1 else "Normal"
    
    print(f"Original Text: {text}")
    print(f"True Label: {actual_str} | Predicted Label: {pred_str}")
    print("-" * 70)

The SVM model misclassified a total of 68651 comments.

Here are 5 typical failure cases:

Original Text: I think a significant amount would be against spending their tax dollars on other people. I bet if that money was poured into college debt or health debt relief, 81% of Americans would have been for it instead.
True Label: Normal | Predicted Label: Sarcastic
----------------------------------------------------------------------
Original Text: why you fail me, my precious? Clinton struggles to gain traction in Florida, despite spending - Clinton shocked money can't buy everything
True Label: Normal | Predicted Label: Sarcastic
----------------------------------------------------------------------
Original Text: At this point they're so stable I could build a house out of em They will never get the stability of the 3DS.
True Label: Normal | Predicted Label: Sarcastic
----------------------------------------------------------------------
Original Text: Jesus is a FNAF fan confirmed Je

## Summary

### Results
- Majority Vote: Achieved an Accuracy of 49.8% and a Macro F1-Score of 0.33. Since our dataset is perfectly balanced, this matches the expected random-guess probability.
- SVM: Achieved an Accuracy of 66.0% and a Macro F1-Score of 0.66. The simple lexical features provided a solid 16% jump in accuracy over the baseline.

### Analysis
- What the SVM learned: By extracting the feature weights, I found that the SVM correctly associates hyperbolic words (e.g., *obviously, clearly, totally*) with sarcasm, and cautious acronyms (e.g., *iirc, afaik*) with normal comments. 
- Where it fails: My error analysis shows that this purely text-based approach often misclassifies comments. It triggers false positives when users use words like "obvious" or "shocked" in a literal, serious context. It also fails to differentiate between general internet humor/exaggeration and actual sarcasm. 

Conclusion: These baseline results confirm that while word frequencies can detect basic sarcasm, our group will definitely need more advanced models that can understand context and semantics to improve accuracy further.